**Viz 6:** Top 10 streets linked to an hour brush (Altair, interactive)

Note: Put an hour-of-day histogram on top. When the user drags a range on it, such as 10 PM–2 AM, the street bar chart below updates. This turns “which streets are worst?” into “which streets are worst when I’m walking home?” It also adds brushing, which is a distinct interaction type.


In [1]:
import altair as alt
import numpy as np
import pandas as pd

# utf-8-sig strips the hidden BOM character at the start of the file
crime = pd.read_csv('crime.csv', encoding='utf-8-sig')
crime.head()

,INCIDENT_NUMBER,OFFENSE_CODE,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,REPORTING_AREA,SHOOTING,OCCURRED_ON_DATE,YEAR,MONTH,DAY_OF_WEEK,HOUR,UCR_PART,STREET,Lat,Long,Location
0,I182079978,619,Larceny,LARCENY ALL OTHERS,C11,817,NaN,6/4/18 9:00,2018,6,Monday,9,Part One,TALBOT AVE,42.287148,-71.064930,"(42.28714820, -71.06493039)"
1,I182079349,727,Auto Theft,AUTO THEFT - LEASED/RENTED VEHICLE,C11,404,NaN,8/12/18 22:00,2018,8,Sunday,22,Part One,VALLEY RD,42.277431,-71.068511,"(42.27743055, -71.06851083)"
2,I182079339,617,Larceny,LARCENY THEFT FROM BUILDING,D4,152,NaN,8/11/18 13:13,2018,8,Saturday,13,Part One,TREMONT ST,42.343934,-71.071688,"(42.34393403, -71.07168782)"
3,I182079317,724,Auto Theft,AUTO THEFT,A7,11,NaN,8/10/18 13:47,2018,8,Friday,13,Part One,WILLIAM F MCCLELLAN HWY,42.393529,-71.011582,"(42.39352862, -71.01158244)"
4,I182079315,724,Auto Theft,AUTO THEFT,A7,11,NaN,7/30/18 14:05,2018,7,Monday,14,Part One,WILLIAM F MCCLELLAN HWY,42.393529,-71.011582,"(42.39352862, -71.01158244)"


### 1. Clean the data and keep incidents near campus

Same rules as Viz 5:
- drop rows with missing or placeholder coordinates (-1 / 0)
- keep only incidents within **0.5 miles** of Snell Library (haversine distance)
- count every offense row (not unique incident numbers)

In [2]:
SNELL_LAT, SNELL_LON = 42.33841, -71.08809   # Snell Library
RADIUS_MI = 0.5
EARTH_RADIUS_MI = 3958.8

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * EARTH_RADIUS_MI * np.arcsin(np.sqrt(a))

df = crime.dropna(subset=['Lat', 'Long', 'STREET'])
df = df[(df['Lat'] > 40) & (df['Long'] < -70)]   # removes -1 / 0 placeholder coordinates

df = df.copy()
df['dist_mi'] = haversine_miles(SNELL_LAT, SNELL_LON, df['Lat'], df['Long'])
near = df[df['dist_mi'] < RADIUS_MI].copy()
near['STREET'] = near['STREET'].str.strip().str.title()   # 'HUNTINGTON AVE' -> 'Huntington Ave'

print(f'{len(near):,} incidents within {RADIUS_MI} mi of Snell on {near["STREET"].nunique()} streets')

549 incidents within 0.5 mi of Snell on 57 streets


### 2. Reorder the hours so the night is continuous

On a normal 0–23 axis, a late-night range like 10 PM–2 AM wraps around midnight and can't be selected in one drag.
So the hour axis **starts at 6 AM** and ends at 5 AM: `hour_pos` 0 = 6 AM, 16 = 10 PM, 18 = midnight, 23 = 5 AM.

Each bar is drawn at the center of its hour (`hour_pos + 0.5`), so a bar counts as selected when the brush covers its middle.

In [3]:
START_HOUR = 6

def hour_label(h):
    if h == 0:
        return '12 AM'
    if h == 12:
        return '12 PM'
    return f'{h % 12} {"AM" if h < 12 else "PM"}'

near['hour_pos'] = (near['HOUR'] - START_HOUR) % 24
near['hour_mid'] = near['hour_pos'] + 0.5
near['hour_label'] = near['HOUR'].apply(hour_label)

# only the columns the chart needs
chart_df = near[['STREET', 'OFFENSE_CODE_GROUP', 'HOUR', 'hour_mid', 'hour_label']]

# labels for the x axis ticks, e.g. 6 -> '12 PM'
tick_labels = {pos: hour_label((pos + START_HOUR) % 24) for pos in range(0, 25, 2)}
label_expr = ' : '.join(f"datum.value == {pos} ? '{lab}'" for pos, lab in tick_labels.items()) + " : ''"

# check the reordering: incidents per hour, in the order they appear on the axis
near.groupby(['hour_mid', 'hour_label']).size().rename('incidents').reset_index().T

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
hour_mid,0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,...,14.5,15.5,16.5,17.5,18.5,19.5,20.5,21.5,22.5,23.5
hour_label,6 AM,7 AM,8 AM,9 AM,10 AM,11 AM,12 PM,1 PM,2 PM,3 PM,...,8 PM,9 PM,10 PM,11 PM,12 AM,1 AM,2 AM,3 AM,4 AM,5 AM
incidents,12,14,27,23,25,15,37,33,34,39,...,28,28,15,18,29,15,14,8,8,1


### 3. Chart

**Drag across the histogram** to pick a time window (for example 10 PM–2 AM). The bottom chart updates to show the
top 10 streets for only those hours. Drag the selection to move it, and double-click to clear it and see all hours again.

In [4]:
HIGHLIGHT = '#1F4E79'
MUTED = '#C9CED6'
CHART_WIDTH = 650

brush = alt.selection_interval(encodings=['x'], name='hours')

# --- Top: hour-of-day histogram with the brush ---
hours = alt.Chart(chart_df).mark_bar(width=CHART_WIDTH / 24 - 3).encode(
    x=alt.X('hour_mid:Q',
            title='Hour of day (starting at 6 AM, so the night stays together)',
            scale=alt.Scale(domain=[0, 24], nice=False),
            axis=alt.Axis(values=list(range(0, 25, 2)), labelExpr=label_expr, grid=False)),
    y=alt.Y('count():Q', title='Incidents'),
    color=alt.condition(brush, alt.value(HIGHLIGHT), alt.value(MUTED)),
    tooltip=[alt.Tooltip('hour_label:N', title='Hour starting'),
             alt.Tooltip('count():Q', title='Incidents')],
).add_params(brush).properties(
    width=CHART_WIDTH, height=160,
    title=alt.Title('When do incidents happen?', subtitle='Drag across the bars to pick the hours you walk home'),
)

# --- Bottom: top 10 streets for the selected hours ---
streets = alt.Chart(chart_df).transform_filter(
    brush
).transform_aggregate(
    incidents='count()', groupby=['STREET']
).transform_joinaggregate(
    total='sum(incidents)'
).transform_calculate(
    share='datum.incidents / datum.total'
).transform_window(
    rank='row_number()',
    sort=[alt.SortField('incidents', order='descending'), alt.SortField('STREET', order='ascending')],
).transform_filter(
    alt.datum.rank <= 10
).mark_bar(color=HIGHLIGHT).encode(
    x=alt.X('incidents:Q', title='Incidents in selected hours'),
    y=alt.Y('STREET:N', title=None,
            sort=alt.EncodingSortField(field='rank', order='ascending')),
    tooltip=[alt.Tooltip('STREET:N', title='Street'),
             alt.Tooltip('incidents:Q', title='Incidents'),
             alt.Tooltip('share:Q', title='Share of selected hours', format='.0%')],
).properties(
    width=CHART_WIDTH, height=280,
    title=alt.Title('Top 10 streets in the selected hours',
                    subtitle='Shows all hours until you make a selection'),
)

labels = streets.mark_text(align='left', dx=4, color='#333').encode(
    text='incidents:Q', tooltip=alt.value(None))

viz6 = alt.vconcat(hours, streets + labels, spacing=30).properties(
    title=alt.Title('Which streets near Northeastern are worst when you walk home?',
                    subtitle='Serious (UCR Part One) incidents within 0.5 miles of Snell Library',
                    fontSize=16, anchor='start'),
)

viz6

alt.VConcatChart(...)

In [5]:
viz6

alt.VConcatChart(...)

In [6]:
# Saved as JSON for the website (where.html loads charts/streets.json)
viz6.save('../charts/streets.json')

## Paragraph outlines

### Takeaway paragraph (webpage)
1. **Question:** which streets are worst *when I'm walking home*?
2. **All hours:** Huntington Ave (15% of incidents) and Mass Ave (13.5%) lead. Huntington is mostly larceny (58 of 83).
3. **10 PM–2 AM:** Mass Ave moves to #1 (18%), followed by Huntington (13%) and Tremont (9%), and Ruggles St enters the top 10.
4. **Overall timing:** the busiest hours are afternoon and early evening (6 PM is the peak), not late night. Night hours make up 42% of incidents.
5. **Caveat:** the 10 PM–2 AM window has only 77 incidents, so the lower-ranked streets aren't meaningful. Busy streets have more people, so counts favor them.

### Design paragraph (Word doc)
- **Why brushing:** a different kind of interaction from the tooltips and zoom in your other charts, which the assignment rewards.
- **Linked views:** the histogram (overview) drives the bar chart (detail).
- **Axis reorder:** starting at 6 AM so a late-night window doesn't wrap around midnight. This is a deliberate design decision, so say so.
- **Color:** dark blue for selected hours, gray for the rest. Bar chart in the same blue to show they're connected.
- **Bar chart choices:** sorted, horizontal so long street names fit, count labels on the bars, ties broken alphabetically.
- **Defaults:** shows all hours until the user makes a selection. The tooltip shows each street's share of the selected window.

## Takeaway paragraph (webpage)

This visualization helps answer the question "which streets near the NEU Boston campus are the most dangerous when I'm walking home?" Across all hours, Huntington Ave and Massachusetts Ave have the most crime, making up about 15% and 13.5% of incidents near campus. Of the 83 incidents on Huntington Ave, most were larceny (58 instances). However, the top streets change depending on the time of day. When selecting 10 PM to 2 AM, Massachusetts Ave moves to the top with about 18% of incidents, followed by Huntington Ave (13%) and Tremont St (9%), and Ruggles St also enters the top 10. Further, the hour chart shows that most crime near campus happens in the afternoon and early evening, with the peak at 6 PM, rather than late at night. However, we must note that the 10 PM to 2 AM window only has 77 incidents, so the streets lower on the list only have a few incidents each and should not be read too closely. Also, busy streets like Huntington Ave and Mass Ave have more people on them, so they may look more dangerous just because of foot traffic. All in all, the key takeaway here is that students walking home late at night should be most aware on Mass Ave, Huntington Ave, and Tremont St, and they can drag across the hour chart to find which streets to watch out for during the exact time they walk home.

## Design paragraph (Word doc)

For this visualization, we used brushing, which is a different interaction from the tooltips and zooming in our other visualizations. The hour-of-day histogram on top gives an overview of when crime happens, and when the user drags across a range of hours, the bar chart below updates to show the top 10 streets for only those hours. We started the hour axis at 6 AM instead of midnight so that late-night windows like 10 PM to 2 AM stay together and can be selected in one drag. Selected hours are shown in dark blue and the rest fade to gray, and the street bars use the same dark blue so it is clear the two charts are connected. Further, we made the street chart horizontal so long street names fit, sorted the bars from most to least incidents, labeled each bar with its count, and broke ties alphabetically so there are always exactly 10 streets. All in all, the chart shows all hours until the user makes a selection, and the tooltip shows each street's share of incidents in the selected hours.